# CNN_5_7ch_Exp18: Cross-loss Hard Voting (Exp13 + Exp17B)

# 1.Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    precision_recall_fscore_support,
)

# 2. Setting

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")

EXP13_DIRS = {
    42: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_seed42",
    43: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_Seed43",
    44: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_Seed44",
}
EXP17_DIRS = {
    42: PROJECT_DIR / "results" / "CNN_5_7ch_Exp17B_Focal_seed42",
    43: PROJECT_DIR / "results" / "CNN_5_7ch_Exp17B_Focal_seed43",
    44: PROJECT_DIR / "results" / "CNN_5_7ch_Exp17B_Focal_seed44",
}

RESULT_DIR = PROJECT_DIR / "results" / "CNN_5_7ch_Exp18_cross_HV"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

STAGES = ["W", "N1", "N2", "N3", "R"]
LABELS = [0, 1, 2, 3, 4]
PROB_COLS = ["prob_" + s for s in STAGES]
SEEDS = [42, 43, 44]

# 3.โหลด CSV และตรวจความตรงกัน

In [ ]:
dfs = {}
for s in SEEDS:
    dfs[f"Exp13_s{s}"] = pd.read_csv(EXP13_DIRS[s] / "val_predictions.csv")
    dfs[f"Exp17B_s{s}"] = pd.read_csv(EXP17_DIRS[s] / "val_predictions.csv")

first = list(dfs.values())[0]

for name, df in dfs.items():
    assert len(df) == len(first), name + ": จำนวนแถวไม่เท่ากัน"
    for col in ["subject", "epoch_start", "epoch_end", "actual"]:
        assert (df[col].astype(str).values == first[col].astype(str).values).all(), name + ": " + col + " ไม่ตรงกัน"

y_true = first["actual"].map(dict(zip(STAGES, LABELS))).to_numpy()
assert not pd.isna(y_true).any(), "พบ actual ที่ไม่อยู่ใน STAGES"
y_true = y_true.astype(int)
probs = {name: df[PROB_COLS].to_numpy() for name, df in dfs.items()}

for name, prob in probs.items():
    assert np.isfinite(prob).all(), f"{name}: พบ NaN/Inf"
    assert np.allclose(prob.sum(axis=1), 1, atol=1e-3), f"{name}: Probability รวมไม่เท่ากับ 1"
    pred = np.array(STAGES)[prob.argmax(axis=1)]
    assert (pred == dfs[name]["predicted"].to_numpy()).all(), f"{name}: predicted ไม่ตรงกับ Probability"

assert len(first) == 2878, f"จำนวน Validation sequences ผิดปกติ: {len(first)}"
print("Validation sequences:", len(first), "| subjects:", first["subject"].nunique(), "| โมเดล:", len(dfs))

exp13_names = ["Exp13_s42", "Exp13_s43", "Exp13_s44"]
exp17_names = ["Exp17B_s42", "Exp17B_s43", "Exp17B_s44"]
all_names = exp13_names + exp17_names

# 4.Hard Voting

In [ ]:
def hard_vote(names):
    votes = np.array([probs[n].argmax(axis=1) for n in names])
    prob_sum = sum(probs[n] for n in names)
    result = []
    ties = 0
    for i in range(votes.shape[1]):
        counts = np.bincount(votes[:, i], minlength=5)
        best = np.where(counts == counts.max())[0]
        if len(best) == 1:
            result.append(best[0])
        else:
            ties += 1
            result.append(best[prob_sum[i][best].argmax()])
    print("เสมอ", ties, "ตัวอย่าง")
    return np.array(result)

print("HV Exp13 (3 โมเดล)")
pred_hv13 = hard_vote(exp13_names)
print("HV Exp17B (3 โมเดล)")
pred_hv17 = hard_vote(exp17_names)
print("HV Cross-loss (6 โมเดล)")
pred_hv6 = hard_vote(all_names)

# 5.คำนวณ metric แสดงผล เปรียบเทียบ

In [ ]:
def get_metrics(y_pred):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)
    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1.mean(),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }
    for i, s in enumerate(STAGES):
        result["f1_" + s] = f1[i]
    result["recall_N1"] = r[1]
    return result

rows = {}
for name in all_names:
    rows[name] = get_metrics(probs[name].argmax(axis=1))
rows["HV Exp13 (3)"] = get_metrics(pred_hv13)
rows["HV Exp17B (3)"] = get_metrics(pred_hv17)
rows["HV Cross-loss (6)"] = get_metrics(pred_hv6)

comparison = pd.DataFrame(rows).T
display(comparison.round(4).style.highlight_max(axis=0, color="lightgreen"))
comparison.to_csv(RESULT_DIR / "val_comparison.csv")

# 6. บันทึกคำทำนายรายตัวอย่าง

In [ ]:
table = first[["subject", "epoch_start", "epoch_end", "actual"]].copy()
for name in all_names:
    table["pred_" + name] = np.array(STAGES)[probs[name].argmax(axis=1)]
table["pred_hv13"] = np.array(STAGES)[pred_hv13]
table["pred_hv17"] = np.array(STAGES)[pred_hv17]
table["pred_hv6"] = np.array(STAGES)[pred_hv6]
table.to_csv(RESULT_DIR / "val_ensemble_predictions.csv", index=False)